we can think this method as a assumption test
* there is no trend or drift is zero
* we calculate the mean and the variance of the mean from zero which is std / np.sqrt(N)
* zscore = mean / (std / np.sqrt(N)), if the zscore in [-0.6, 0.6], no signficant from zero and thus is neutral regime. but when abs[zscore] in [0.6, 1.7], then it is in quiet trending regime, but if it is more than 1.7, then it is in volatile regime means the move is too significant from zero. 

The longer the look back horizon, the more long trending we are looking for. 

Compared to the ma_trending.trend_score, they are similar but ma_trend is looking the accumulated drift and normalize with noise to check if it is significant from zero to detect how strong the trending is. We can have similar trending score there to avoid put too much risk during bull volatile regime. But the trend_score normalized with atr which is larger than based on close and thus is typically hard to get more than 1.0

at 20w window
1. spy tstats average was 0.3 while xlk was about 0.5

In [1]:
from sts.dio.equity import Ticker
from sts.quant.candle import Candle
from sts.quant.indicators.trend import ma_trend
import numpy as np
from IPython import display
import pandas as pd

In [15]:
ticker = Ticker("ASML")
df2 = Candle(ticker.history(period="1y"))

In [16]:
df = Candle(df2.resample("1D").tail(500))

In [17]:
window = 20  # condition on longer horizon, and then switch to shorter horizon, use that as a signal, as long as longer horizon is back to normal, we should switch back to longer horizon again. horizon (20, 10, 5)
close_diff = df["Close"].diff()
std = close_diff.rolling(window).std()  # volatility does not change too much, it is long term average std
mean = close_diff.rolling(
    window
).mean()  # while mean is the average return over long term, the longer the window, the more closer to the daily drift, but why it scale by square root? not by linear?
SQN = mean / (std / np.sqrt(window))  # because the std of mean to the actual drift is scaled by std / np.sqrt(nobs).
trend_score = ma_trend.trend_score(
    df
)  # looks the trend score is also good: it capture the long term trend 0.3 for spy, with amplitude 0.6

In [18]:
pd.options.plotting.backend = "plotly"
result = SQN.quantile([0.1, 0.3, 0.5, 0.7, 0.9])
display.display(
    pd.Series(
        result.values,
        index=["BearVolatile", "BearQuiet", "Neutral", "BullQuiet", "BullVolatile"],
    )
    .to_frame()
    .T.round(2)
)
SQN.plot()

,BearVolatile,BearQuiet,Neutral,BullQuiet,BullVolatile
0,-1.01,-0.53,0.08,0.51,1.01


In [14]:
df.plot()

In [7]:
df["Volume"].plot()